# 面试问题：GraphRAG 怎样区分 Local Search 与 Global Search，并保持图谱来源可核验？

**一句话回答。** Local Search 围绕命中的实体、关系和原文 chunk 回答具体问题；Global Search 对社区报告做 map-reduce 汇总回答全局主题。两条路径都必须保留 edge/claim 的 source chunk、图谱版本与权限，社区摘要不能成为无来源事实。

本 Notebook 用 Python 标准库手写最小数据合同、状态机、验证器和失败分支。断言针对受控小数据，不等于模型语义正确、数据库安全、图谱质量或生产 Agent 的安全保证。

**资料入口。** [Microsoft GraphRAG Query Engine](https://microsoft.github.io/graphrag//query/overview/) 区分实体型 Local Search 与社区报告型 Global Search；本例用小图展示来源和预算合同。


In [ ]:
question = "GraphRAG local global search"  # 执行本行的状态、计算或校验逻辑。
assert "GraphRAG" in question  # 执行本行的状态、计算或校验逻辑。
assert 6 / 2 == 3  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 图谱 extraction 产物需要 entity、edge 和原文来源三层

entity 名称、关系三元组和 claim 都可能由模型抽取而出，因此不能只存图边。每条边应保留 source chunk、抽取器版本、置信度/审核状态和 ACL，方便误抽取时精确回滚。


In [ ]:
chunks = {"c1": "Alice 负责退款审批", "c2": "退款流程需要人工确认", "c3": "Bob 负责配送时效"}  # 执行本行的状态、计算或校验逻辑。
entities = {"alice": {"name": "Alice", "kind": "person"}, "refund": {"name": "退款", "kind": "process"}, "bob": {"name": "Bob", "kind": "person"}}  # 执行本行的状态、计算或校验逻辑。
edges = [{"source": "alice", "relation": "负责", "target": "refund", "chunk": "c1", "extractor": "g1"}, {"source": "refund", "relation": "要求", "target": "人工确认", "chunk": "c2", "extractor": "g1"}, {"source": "bob", "relation": "负责", "target": "配送", "chunk": "c3", "extractor": "g1"}]  # 执行本行的状态、计算或校验逻辑。
assert len(edges) == 3  # 执行本行的状态、计算或校验逻辑。
assert all(edge["chunk"] in chunks for edge in edges)  # 执行本行的状态、计算或校验逻辑。
assert all(edge["extractor"] == "g1" for edge in edges)  # 执行本行的状态、计算或校验逻辑。

## 2. Local Search 先命中实体，再展开邻居和原文

具体问题如“谁负责退款”需要实体邻域而不是整库主题摘要。真实实现可混合 embedding、图 traversal 和 chunk retriever；教学先按实体名匹配，并限制 hop/budget 防止高连接节点吞没上下文。


In [ ]:
def local_search(query, hop_limit):  # 执行本行的状态、计算或校验逻辑。
    starts = [entity_id for entity_id, entity in entities.items() if entity["name"] in query]  # 执行本行的状态、计算或校验逻辑。
    selected = [edge for edge in edges if edge["source"] in starts][:hop_limit]  # 执行本行的状态、计算或校验逻辑。
    return starts, selected  # 执行本行的状态、计算或校验逻辑。
starts, local_edges = local_search("Alice 如何处理", 2)  # 执行本行的状态、计算或校验逻辑。
assert starts == ["alice"]  # 执行本行的状态、计算或校验逻辑。
assert local_edges[0]["chunk"] == "c1"  # 执行本行的状态、计算或校验逻辑。
assert len(local_edges) == 1  # 执行本行的状态、计算或校验逻辑。

## 3. Local answer 的证据必须回到 chunk，而非只说图边

图边是抽取结果，原文是用户可阅读的依据。若 edge 缺失 source、被审核否决或版本过期，Local Search 应降低置信、回退 raw retrieval 或拒答，而不是凭 entity label 编造关系。


In [ ]:
def local_evidence(selected_edges):  # 执行本行的状态、计算或校验逻辑。
    return [{"edge": edge["relation"], "chunk": edge["chunk"], "raw": chunks[edge["chunk"]]} for edge in selected_edges]  # 执行本行的状态、计算或校验逻辑。
local_evidence_rows = local_evidence(local_edges)  # 执行本行的状态、计算或校验逻辑。
assert local_evidence_rows[0]["raw"] == "Alice 负责退款审批"  # 执行本行的状态、计算或校验逻辑。
assert local_evidence_rows[0]["chunk"] == "c1"  # 执行本行的状态、计算或校验逻辑。
assert "Alice" in local_evidence_rows[0]["raw"]  # 执行本行的状态、计算或校验逻辑。

## 4. 社区报告面向跨数据集主题，而不是实体问答

Global Search 聚合社区报告，适合“主要主题是什么”一类问题。社区报告需要 members、source chunks、层级和生成版本；map 阶段各报告产出带分数的 point，reduce 在 token 预算内选择点。


In [ ]:
reports = [{"id": "r_refund", "summary": "退款主题包含审批与人工确认", "members": ("alice", "refund"), "chunks": ("c1", "c2"), "level": 1, "version": "g1"}, {"id": "r_delivery", "summary": "配送主题包含负责人与时效", "members": ("bob",), "chunks": ("c3",), "level": 1, "version": "g1"}]  # 执行本行的状态、计算或校验逻辑。
assert len(reports) == 2  # 执行本行的状态、计算或校验逻辑。
assert all(report["chunks"] for report in reports)  # 执行本行的状态、计算或校验逻辑。
assert all(report["version"] == "g1" for report in reports)  # 执行本行的状态、计算或校验逻辑。

## 5. map-reduce 需要显式 point 分数和上下文预算

本例用字符重叠代替 LLM map。生产 map 可以输出带 importance 的结构化 point，但 reduce 必须只消费有来源的 point，并报告被截断报告/点，不能把全局回答当作无成本的一次生成。


In [ ]:
def report_score(query, report):  # 执行本行的状态、计算或校验逻辑。
    return len(set(query) & set(report["summary"]))  # 执行本行的状态、计算或校验逻辑。
def global_search(query, budget):  # 执行本行的状态、计算或校验逻辑。
    points = [{"report": report["id"], "score": report_score(query, report), "chunks": report["chunks"]} for report in reports]  # 执行本行的状态、计算或校验逻辑。
    return [point for point in sorted(points, key=lambda point: point["score"], reverse=True) if point["score"] > 0][:budget]  # 执行本行的状态、计算或校验逻辑。
points = global_search("退款和配送主题", 2)  # 执行本行的状态、计算或校验逻辑。
assert {point["report"] for point in points} == {"r_refund", "r_delivery"}  # 执行本行的状态、计算或校验逻辑。
assert all(point["score"] > 0 for point in points)  # 执行本行的状态、计算或校验逻辑。
assert len(points) == 2  # 执行本行的状态、计算或校验逻辑。

## 6. Global answer 同样需要展开到原文来源

社区 summary 可作为组织线索，最终回答应给出每个 point 所属 source chunks，必要时重取 raw 文本。默认不允许 general knowledge 混入私有数据回答，否则来源边界和幻觉评测会失真。


In [ ]:
def global_evidence(point_values):  # 执行本行的状态、计算或校验逻辑。
    return sorted({chunk_id for point in point_values for chunk_id in point["chunks"]})  # 执行本行的状态、计算或校验逻辑。
global_chunks = global_evidence(points)  # 执行本行的状态、计算或校验逻辑。
assert global_chunks == ["c1", "c2", "c3"]  # 执行本行的状态、计算或校验逻辑。
assert all(chunk_id in chunks for chunk_id in global_chunks)  # 执行本行的状态、计算或校验逻辑。
assert "c2" in global_chunks  # 执行本行的状态、计算或校验逻辑。

## 7. 图谱版本与权限决定索引是否可用

修改 extraction prompt、实体归一化、社区算法或原文权限后都可能改变图。必须整体或分区重建并把 query 绑定图谱版本；跨租户 community report 是高风险泄漏点，不能仅靠生成阶段提示词处理。


In [ ]:
def graph_compatible(report, graph_version):  # 执行本行的状态、计算或校验逻辑。
    return report["version"] == graph_version  # 执行本行的状态、计算或校验逻辑。
assert graph_compatible(reports[0], "g1")  # 执行本行的状态、计算或校验逻辑。
assert not graph_compatible(reports[0], "g2")  # 执行本行的状态、计算或校验逻辑。
assert all(report["level"] == 1 for report in reports)  # 执行本行的状态、计算或校验逻辑。

## 8. 评测将 local、global 和抽取质量拆开

至少分别测实体 linking/edge precision、local evidence recall、global theme coverage、community summary faithfulness、成本和延迟。Global 失败可能源于抽取/社区/摘要/map/reduce 任一层，不能只调整最终 prompt。


In [ ]:
def coverage(required, observed):  # 执行本行的状态、计算或校验逻辑。
    return set(required).issubset(set(observed))  # 执行本行的状态、计算或校验逻辑。
assert coverage(["c1"], [row["chunk"] for row in local_evidence_rows])  # 执行本行的状态、计算或校验逻辑。
assert coverage(["c1", "c2", "c3"], global_chunks)  # 执行本行的状态、计算或校验逻辑。
assert not coverage(["c4"], global_chunks)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试先按问题类型选路：具体实体关系走 Local，跨库主题走 Global；再讲抽取 edge 的来源、community report 的 map-reduce 预算、raw evidence 展开、版本/ACL 和分层指标。GraphRAG 不是“抽完三元组就让模型自由总结”。
